# MELG19937 Full-Period Search with Optimized Lagged Tempering

Search for full-period MELG generators with $k = 19937$ (period $2^{19937}-1$),
then optimize the Lagged Tempering parameters for each one using the
backtracking algorithm (Harase 2009) with the lattice method.

Parameters: $w=64$, $N=312$, $r=31$, $k = N \times w - r = 312 \times 64 - 31 = 19937$.

Each section loads its data from disk, so sections can be run independently.

## Constants

In [ ]:
GENERATORS_FILE = "../../yaml/generators/MELG/N312_r31_w64.yaml"
TESTED_DIR = "../../yaml/testedgenerators"
FAMILY = "MELG"
W = 64
STRUCTURAL_PARAMS = {"w": 64, "N": 312, "r": 31}

## 1. Search for full-period MELG19937 generators

Structural: `w=64`, `N=312`, `r=31`.  Randomized: `M`, `sigma1`, `sigma2`, `a`.

In [ ]:
from regpoly.search.search_primitive import PrimitiveSearch

search = PrimitiveSearch(
    family=FAMILY,
    L=W,
    structural_params=STRUCTURAL_PARAMS,
    fixed_params={},
    max_tries=100000,
    max_seconds=3600 * 12,
    generators_dir="../../yaml/generators",
)

print(f"Output file: {search.output_file}")
results = search.run()

## 2. Load found generators

In [ ]:
import yaml

with open(GENERATORS_FILE) as f:
    data = yaml.safe_load(f)

family = data["family"]
common = data["common"]
gen_entries = data["generators"]
n_gens = len(gen_entries)
arr_size = common["N"] - 1

print(f"Loaded {n_gens} full-period MELG19937 generators from {GENERATORS_FILE}")
print(f"Structural: {common}")
print()
for i, entry in enumerate(gen_entries[:10], 1):
    print(f"  {i}. M={entry['M']}, sigma1={entry['sigma1']}, "
          f"sigma2={entry['sigma2']}, a=0x{entry['a']:016x}")
if n_gens > 10:
    print(f"  ...and {n_gens - 10} more")

## 3. Optimize Lagged Tempering for each generator

For each full-period generator, run the backtracking optimizer with
different random starting parameters (`sigma`, `L`, `b`).  Uses the
lattice method for equidistribution computation (required for k=19937).
Save generators that reach $\Delta = 0$ (ME) or low $\Delta$.

In [ ]:
import random
import time
import yaml
from regpoly.core.generator import Generator
from regpoly.core.transformation import Transformation
from regpoly.core.combination import Combination
from regpoly.search.tempering_optimizer import TemperingOptimizer
from regpoly.io.tested_generator import save_tested_generator

N_OPT_TRIES = 10
MAX_BACKTRACKS = 300

# Reload generators from disk
with open(GENERATORS_FILE) as f:
    data = yaml.safe_load(f)

family = data["family"]
common = data["common"]
gen_entries = data["generators"]
n_gens = len(gen_entries)
w = common["w"]
arr_size = common["N"] - 1

print(f"Optimizing {n_gens} generators x {N_OPT_TRIES} tries")
print(f"Method: lattice, max_backtracks: {MAX_BACKTRACKS}")
print()

t_total = time.time()

for i, entry in enumerate(gen_entries):
    gen = Generator.create(family, w, **common, **entry)
    print(f"Generator {i+1}/{n_gens}: M={entry['M']}, "
          f"sigma1={entry['sigma1']}, sigma2={entry['sigma2']}, "
          f"a=0x{entry['a']:016x}")

    best_se = None
    best_params = None

    for t in range(N_OPT_TRIES):
        sigma = random.randint(1, w - 1)
        L = random.randint(1, arr_size - 2)
        b = random.getrandbits(w)

        trans = Transformation.create("laggedTempering",
                               w=w, sigma=sigma, L=L, b=b)

        comb = Combination(J=1, Lmax=w)
        comb.components[0].add_gen(gen)
        comb.components[0].add_trans(trans)
        comb.reset()

        opt = TemperingOptimizer(comb, max_essais=MAX_BACKTRACKS,
                                 verbose=False)
        result = opt.run(comb)

        status = "ME!" if result.se == 0 else f"se={result.se}"
        print(f"  try {t+1:>2d}: sigma={sigma:>2d}, L={L:>3d} "
              f"-> {status} ({result.elapsed:.1f}s, "
              f"{result.essais} bt)")

        if best_se is None or result.se < best_se:
            best_se = result.se
            best_params = {
                "sigma": sigma, "L": L,
                "b": trans.get_param("b"),
            }

        if result.se == 0:
            test_results = {
                "equidistribution": {"se": 0, "status": "ME"}
            }
            path = save_tested_generator(TESTED_DIR, "equidist", comb,
                                         test_results)
            print(f"    -> Saved: {path}")
            break

    if best_se is not None and best_se > 0:
        print(f"  Best: se={best_se} (sigma={best_params['sigma']}, "
              f"L={best_params['L']})")

    print()

elapsed_total = time.time() - t_total
print(f"\nDone in {elapsed_total:.0f}s")

## 4. Results summary

Scan the tested generators directory for all MELG19937 results.

In [ ]:
import glob
import yaml

pattern = f"{TESTED_DIR}/MELG/N312_r31_w64.equidist.*.yaml"
files = sorted(glob.glob(pattern))

print(f"Found {len(files)} tested MELG19937 generators")
if not files:
    print("No results yet. Run section 3 first.")
else:
    print(f"\n{'#':<4} {'M':<4} {'s1':<4} {'s2':<4} {'a':<20} "
          f"{'sig':<4} {'L':<4} {'se':>4}  {'file'}")
    print("-" * 85)
    for rank, filepath in enumerate(files, 1):
        with open(filepath) as f:
            d = yaml.safe_load(f)
        g = d["generator"]
        t = d["tempering"][0] if d.get("tempering") else {}
        r = d.get("results", {}).get("equidistribution", {})
        se = r.get("se", "?")
        print(f"{rank:<4} {g.get('M','?'):<4} {g.get('sigma1','?'):<4} "
              f"{g.get('sigma2','?'):<4} 0x{g.get('a',0):016x} "
              f"{t.get('sigma','?'):<4} {t.get('L','?'):<4} {se:>4}  "
              f"{filepath.split('/')[-1]}")

## 5. Verify a tested generator

Load a tested generator from disk and recompute its equidistribution.

In [ ]:
import glob
from regpoly.tested_generator import load_tested_generator
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest, METHOD_DUALLATTICE
)

INT_MAX = 2**31 - 1

# Pick the first tested MELG19937 generator
pattern = f"{TESTED_DIR}/MELG/N312_r31_w64.equidist.*.yaml"
files = sorted(glob.glob(pattern))

if not files:
    print("No tested generators found. Run section 3 first.")
else:
    filepath = files[0]
    print(f"Loading: {filepath}")
    comb, saved_results = load_tested_generator(filepath)

    print(f"  Generator: {comb[0].display()}")
    print(f"  Saved results: {saved_results}")
    print()

    print("Recomputing equidistribution (lattice method)...")
    test = EquidistributionTest(
        L=comb.L,
        delta=[INT_MAX] * (comb.L + 1),
        mse=INT_MAX,
        method=METHOD_DUALLATTICE,
    )
    result = test.run(comb)

    print(f"Dimension gaps sum = {result.se}")
    if result.is_me():
        print("==> MAXIMALLY EQUIDISTRIBUTED!")
    print()

    table_str, _ = result.display_table(comb, 'l')
    print(table_str)